<a href="https://colab.research.google.com/github/Keerthika11-eng/keerthika11-eng/blob/main/hybrid-dataset-processing-pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [ ]:
!pip install pyspark
!pip install faker
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.6 MB/s eta 0:00:00


Import Libraries & Initialize Faker


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import uuid
import hashlib
import random

# Initialize Faker with US English locale
fake = Faker('en_US')

Define Hash Function

In [ ]:
def sha256_hash(value):
    """Returns a SHA-256 hash of a normalized string."""
    return hashlib.sha256(str(value).strip().lower().encode()).hexdigest()

def generate_hybrid_dataset(total_rows=10000, unique_count=1000):
    data = []

    # Generate a master pool of IDs to ensure we don't run out
    id_pool = random.sample(range(10001, 999999), 6000)

    # 1. Create the 1,000 UNIQUE records (Loners)
    for i in range(unique_count):
        raw_email = fake.email()
        raw_maid = fake.uuid4()
        raw_phone=fake.phone_number()

        data.append({
            "record_id": id_pool[i],
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": raw_email,
            "house_number": fake.building_number(),
            "street": fake.street_name(),
            "city": fake.city(),
            "zip_code": fake.zipcode(),
            "ip_address": fake.ipv4(),
            "maid":raw_maid,
            "phone":raw_phone,
            "email_sha256": sha256_hash(raw_email),
            "phone_sha256": sha256_hash(fake.phone_number()),
            "maid_sha256": sha256_hash(str(uuid.uuid4()))
        })

    # 2. Create the remaining 9,000 rows using DUPLICATE ID groups
    current_row_count = unique_count
    id_idx = unique_count

    while current_row_count < total_rows:
        group_id = id_pool[id_idx]
        group_size = random.choice([2, 3]) # Every duplicate ID appears 2 or 3 times

        if current_row_count + group_size > total_rows:
            group_size = total_rows - current_row_count

        for _ in range(group_size):

            raw_email = fake.email()
            raw_maid = fake.uuid4()
            raw_phone=fake.phone_number()

            data.append({
                "record_id": group_id,
                "first_name": fake.first_name(),
                "last_name": fake.last_name(),
                "email": raw_email,
                "house_number": fake.building_number(),
                "street": fake.street_name(),
                "city": fake.city(),
                "zip_code": fake.zipcode(),
                "ip_address": fake.ipv4(),
                "maid":raw_maid,
                "phone":raw_phone,
                "email_sha256": sha256_hash(fake.email()),
                "phone_sha256": sha256_hash(fake.phone_number()),
                "maid_sha256": sha256_hash(str(uuid.uuid4()))
            })
            current_row_count += 1
        id_idx += 1

    # Shuffle the deck so unique and duplicate rows are mixed randomly
    df = pd.DataFrame(data).sample(frac=1).reset_index(drop=True)
    return df

Generate Hybrid Dataset (Unique + Duplicate Records)

In [ ]:
# Generate the data
df = generate_hybrid_dataset(10000, 1000)

# Statistics Verification
counts = df['record_id'].value_counts()
print(f"Total Rows: {len(df)}")
print(f"Unique IDs (Loners): {sum(counts == 1)}")
print(f"IDs with 2+ records: {sum(counts > 1)}")

# Save the dataframe to a CSV file in the Colab environment
df.to_csv('hybrid_dataset.csv', index=False)
print("\nSuccess: Dataset saved as 'hybrid_dataset.csv'")

# View a small sample
df[['record_id', 'first_name', 'last_name',"email","phone"]].head(10)

Total Rows: 10000
Unique IDs (Loners): 1001
IDs with 2+ records: 3574

Success: Dataset saved as 'hybrid_dataset.csv'


,record_id,first_name,last_name,email,phone
0,743726,Ryan,Martinez,loweryjennifer@example.com,(417)641-6092
1,725646,Amy,Briggs,piercejaime@example.com,(245)921-0619
2,774079,Stephanie,Carter,gabriel32@example.net,(420)617-0914
3,171646,Kellie,George,danderson@example.net,2932830936
4,617665,Dawn,Walker,sarahsnyder@example.net,367.865.6854x80975
5,800508,Joseph,Lin,james77@example.net,7182184318
6,320910,Jamie,May,xturner@example.org,810.884.4524
7,471106,James,Rodriguez,matthewsims@example.org,364.687.1652x362
8,420690,Michelle,Mcdonald,wagnermichelle@example.net,880-607-9265
9,411931,Daniel,Weaver,reevesmatthew@example.com,863.523.9925x326


Create Name Master Table

In [ ]:
name_master = df[['record_id','first_name','last_name']]

name_master.to_csv("name_master.csv", index=False)

print(name_master.head())

   record_id first_name last_name
0     743726       Ryan  Martinez
1     725646        Amy    Briggs
2     774079  Stephanie    Carter
3     171646     Kellie    George
4     617665       Dawn    Walker


Create Email Master Table

In [ ]:
email_master = df[['record_id','email','email_sha256']]

email_master.to_csv("email_master.csv", index=False)

print(email_master.head())

   record_id                       email  \
0     743726  loweryjennifer@example.com   
1     725646     piercejaime@example.com   
2     774079       gabriel32@example.net   
3     171646       danderson@example.net   
4     617665     sarahsnyder@example.net   

                                        email_sha256  
0  2a558d6527d060579b4605bdd7c5c2c4315468c336d1a7...  
1  40e1811ce1daf2d835d955d2146f255bccfa87120474a2...  
2  7a8ceedec5c9b7dbdaa953317fae7c5d2237d4e5a86f51...  
3  88e5b9c2e4f159827418232b3d447d1f878864bded3afc...  
4  45cdaeae687adfb65d23d52f132111f0fef3572ee66ea2...  


Create IP Master Table

In [ ]:
ip_master = df[['record_id','ip_address']]

ip_master.to_csv("ip_master.csv", index=False)

print(ip_master.head())

   record_id      ip_address
0     743726    44.218.89.74
1     725646    86.44.179.57
2     774079   130.75.62.128
3     171646   177.71.139.71
4     617665  119.45.139.254


Create MAID Master Table

In [ ]:
maid_master = df[['record_id','maid', 'maid_sha256']]

maid_master.to_csv("maid_master.csv", index=False)

print(maid_master.head())

   record_id                                  maid  \
0     743726  f5b6b206-723b-4b8e-8e11-c9f2309c66b1   
1     725646  c585a66a-187c-4009-997a-15256be927de   
2     774079  874abbb5-a5e5-44fb-bb6e-5f0e1969ce72   
3     171646  26d91fb5-1bbd-4c37-a6df-e92f8eff2c47   
4     617665  456b6dfa-3771-4fce-ae8d-db1a74959d0a   

                                         maid_sha256  
0  e0c625bc15196f574b726e6b4e5c7d5484f6a2a2a23425...  
1  1907a2dc0c652d7f4360f085799136030acc3ccadbed81...  
2  3ac48ccf8fc96bba8b5c7c4fd3b40ac2e9898c2857e3d9...  
3  50826f858387a67b67a8f592dd4ee481c69325e848a9a9...  
4  e77d33af3af191e9893c731a497032355e27e354880cbf...  


Create Customer Input File

In [ ]:
input_df = pd.DataFrame({
    "customer_id":["cust_001"],
    "ip":["44.218.89.74"] # Changed to a matching IP from ip_master.csv
})

input_df.to_csv("customer_input.csv",index=False)

input_df

,customer_id,ip
0,cust_001,44.218.89.74


Load Customer Data into Spark

In [ ]:
input_path = "customer_input.csv"   # folder

customer_df = spark.read.csv(
    input_path,
    header=True,
    inferSchema=True
)

customer_df.show()

+-----------+------------+
|customer_id|          ip|
+-----------+------------+
|   cust_001|44.218.89.74|
+-----------+------------+



Perform IP-Based Lookup (Join Operation)

In [ ]:
from pyspark.sql.functions import col

# Ensure customer_df is correctly loaded from customer_input.csv
# This re-reads the CSV to prevent any potential variable scope/re-assignment issues
customer_df_from_input = spark.read.csv(
    "customer_input.csv",
    header=True,
    inferSchema=True
)

# Rename the 'ip' column to 'ip_address' in customer_df_from_input
# to match the column in ip_master_df for the join
input_spark_for_join = customer_df_from_input.withColumnRenamed("ip", "ip_address")

ip_master_df = spark.read.csv("ip_master.csv", header=True, inferSchema=True)

# Perform the join
# Select specific columns to avoid duplicates and ensure clarity.
# We want customer_id from input, the common ip_address, and record_id from ip_master_df.
cibil_lookup = input_spark_for_join.join(
    ip_master_df,
    on="ip_address", # Now the join column matches
    how="left"
).select(
    col("customer_id"), # Select customer_id from the input DataFrame
    col("ip_address"),  # Select the common join column
    col("record_id").alias("matched_record_id") # Select record_id from ip_master_df and rename it
)

cibil_lookup.show()

+-----------+------------+-----------------+
|customer_id|  ip_address|matched_record_id|
+-----------+------------+-----------------+
|   cust_001|44.218.89.74|           743726|
+-----------+------------+-----------------+



Join All Master Tables

In [ ]:
name_master_spark = spark.createDataFrame(name_master)
email_master_spark = spark.createDataFrame(email_master)

# Recreate maid_master_spark from the updated pandas maid_master (which now contains maid_sha256)
maid_master_spark = spark.createDataFrame(maid_master)

# Rename 'matched_record_id' to 'record_id' in cibil_lookup for consistent joining
cibil_lookup_renamed = cibil_lookup.withColumnRenamed("matched_record_id", "record_id")

joined_df = cibil_lookup_renamed \
    .join(name_master_spark, "record_id", "left") \
    .join(email_master_spark, "record_id", "left") \
    .join(maid_master_spark, "record_id", "left")

joined_df.show(100)

+---------+-----------+------------+----------+---------+--------------------+--------------------+--------------------+--------------------+
|record_id|customer_id|  ip_address|first_name|last_name|               email|        email_sha256|                maid|         maid_sha256|
+---------+-----------+------------+----------+---------+--------------------+--------------------+--------------------+--------------------+
|   743726|   cust_001|44.218.89.74|     Laura|   Pierce|jadecook@example.org|8917ee04612ee225c...|974f3a84-3c18-436...|f5fba91fcaed7bf81...|
|   743726|   cust_001|44.218.89.74|     Laura|   Pierce|jadecook@example.org|8917ee04612ee225c...|f5b6b206-723b-4b8...|e0c625bc15196f574...|
|   743726|   cust_001|44.218.89.74|     Laura|   Pierce|loweryjennifer@ex...|2a558d6527d060579...|974f3a84-3c18-436...|f5fba91fcaed7bf81...|
|   743726|   cust_001|44.218.89.74|     Laura|   Pierce|loweryjennifer@ex...|2a558d6527d060579...|f5b6b206-723b-4b8...|e0c625bc15196f574...|
|   74

Validate Required PII Columns

In [ ]:
required_pii = ["ip"]

missing_cols = [c for c in required_pii if c not in customer_df.columns]

if len(missing_cols) > 0:
    raise Exception(f"Missing PII columns: {missing_cols}")

Fix Join Using Broadcast (Optimization)

In [ ]:
from pyspark.sql.functions import broadcast, col

# customer_df has columns: 'customer_id', 'ip'
# ip_master_df (Spark DataFrame) has columns: 'record_id', 'ip_address'

# Rename 'ip' in customer_df to 'ip_address' to match ip_master_df for the join
customer_df_renamed_ip = customer_df.withColumnRenamed("ip", "ip_address")

cibil_lookup_fixed = customer_df_renamed_ip.join(
    broadcast(ip_master_df), # Use the Spark DataFrame version of ip_master
    on="ip_address",         # Join on the consistent column name
    how="left"
)

cibil_lookup_fixed.show()

+-------------+-----------+---------+
|   ip_address|customer_id|record_id|
+-------------+-----------+---------+
|102.35.636.96|   cust_001|     NULL|
+-------------+-----------+---------+



Track Source File Information

In [ ]:
from pyspark.sql.functions import input_file_name

customer_df = spark.read.csv("customer_input.csv", header=True, inferSchema=True) \
    .withColumn("source_file", input_file_name())

Detect Latest Order Specification File

In [ ]:
import os

orderspec_files = [f for f in os.listdir() if "orderspec" in f]

if not orderspec_files:
    raise Exception("No OrderSpec found")

latest_orderspec = sorted(orderspec_files)[-1]

print("Using OrderSpec:", latest_orderspec)

Using OrderSpec: orderspec_20260327_082525.csv


Create Dynamic Order Specification

In [ ]:
import pandas as pd
import datetime

orderspec_data = {
    "Description": [
        "First_name",
        "Email",
        "encrypted email",
        "Maid"
    ],
    "Column": [
        "first_name",
        "raw_email",
        "email_sha256",
        "maid_sha256"
    ]
}

orderspec_df = pd.DataFrame(orderspec_data)

# Generate a dynamic filename with a timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
dynamic_orderspec_filename = f"orderspec_{timestamp}.csv"

orderspec_df.to_csv(dynamic_orderspec_filename, index=False)

print(f"Orderspec file '{dynamic_orderspec_filename}' created.")

Orderspec file 'orderspec_20260327_082653.csv' created.


Load Order Specification into Spark

In [ ]:
import os

orderspec_files = [f for f in os.listdir() if "orderspec" in f]

if not orderspec_files:
    raise Exception("No OrderSpec found")

latest_orderspec = sorted(orderspec_files)[-1]

print("Using OrderSpec:", latest_orderspec)

Using OrderSpec: orderspec_20260327_082525.csv


Map Columns Based on OrderSpec

In [ ]:
import pandas as pd

orderspec_data = {
    "Description": [
        "First_name",
        "Email",
        "encrypted email",
        "Maid"
    ],
    "Column": [
        "first_name",
        "raw_email",
        "email_sha256",
        "maid_sha256"
    ]
}

orderspec_df = pd.DataFrame(orderspec_data)

orderspec_df.to_csv("0001_1256_orderspec.csv", index=False)

orderspec_df

,Description,Column
0,First_name,first_name
1,Email,raw_email
2,encrypted email,email_sha256
3,Maid,maid_sha256


Map Columns Based on OrderSpec

In [ ]:
orderspec = spark.read.csv(
    latest_orderspec, # Use the dynamically identified latest_orderspec from the previous cell
    header=True,
    inferSchema=True
)

orderspec.show()

+---------------+------------+
|    Description|      Column|
+---------------+------------+
|     First_name|  first_name|
|          Email|   raw_email|
|encrypted email|email_sha256|
|           Maid| maid_sha256|
+---------------+------------+



Generate Final Output Data

In [ ]:
from pyspark.sql.functions import col

# The original code produced an error because of ambiguous column names after multiple joins
# (e.g., 'first_name', 'email', 'email_sha256' appearing multiple times).

# The PII columns are available in the joined_df after merging with master dataframes.
# We will select from joined_df to get the requested columns.

# Create a list of Spark Column expressions (strings for selectExpr), aliasing them
# to the desired output names from the 'Description' column in orderspec.
selected_spark_expressions = []

# Map from 'Column' in orderspec to the actual column name in joined_df
# 'raw_email' in orderspec maps to 'email' in joined_df
# Note: 'maid_sha256' must be present in joined_df (ensured by modifying U5e_3rG6hDjS)
column_map_to_joined_df = {
    "first_name": "first_name",
    "raw_email": "email",
    "email_sha256": "email_sha256",
    "maid_sha256": "maid_sha256"
}

for row in orderspec.collect():
    description_name = row['Description']
    column_hint = row['Column'] # This is 'first_name', 'raw_email', etc.

    if column_hint in column_map_to_joined_df:
        original_joined_column_name = column_map_to_joined_df[column_hint]
        selected_spark_expressions.append(f"`{original_joined_column_name}` AS `{description_name}`")
    else:
        # Fallback for any other columns, though not expected for this specific orderspec
        selected_spark_expressions.append(f"`{column_hint}` AS `{description_name}`")

# Select from joined_df, as it contains all the necessary PII columns
output_df = joined_df.selectExpr(*selected_spark_expressions)
output_df.show()

+----------+-----+---------------+----+
|First_name|Email|encrypted email|Maid|
+----------+-----+---------------+----+
|      NULL| NULL|           NULL|NULL|
+----------+-----+---------------+----+



Display Final Output

In [ ]:
from pyspark.sql.functions import col

# Get the list of column hints from orderspec, e.g., ['first_name', 'raw_email', 'email_sha256', 'maid_sha256']
columns_hint = [row['Column'] for row in orderspec.collect()]

# Get the list of desired output descriptions, e.g., ['First_name', 'Email', 'encrypted email', 'Maid']
description_names = [row['Description'] for row in orderspec.collect()]

# Map from 'Column' in orderspec to the actual column name in the joined_df
# 'raw_email' in orderspec maps to 'email' in joined_df
column_map_to_joined_df = {
    "first_name": "first_name",
    "raw_email": "email",
    "email_sha256": "email_sha256",
    "maid_sha256": "maid_sha256"
}

selected_spark_expressions = []
for i, column_hint in enumerate(columns_hint):
    description_name = description_names[i]

    if column_hint in column_map_to_joined_df:
        original_joined_column_name = column_map_to_joined_df[column_hint]
        selected_spark_expressions.append(f"`{original_joined_column_name}` AS `{description_name}`")
    else:
        # Fallback for any other columns, though not expected for this specific orderspec
        selected_spark_expressions.append(f"`{column_hint}` AS `{description_name}`")

# Select from joined_df, as it contains all the necessary PII columns
output_df = joined_df.selectExpr(*selected_spark_expressions)
output_df.show()

+----------+--------------------+--------------------+--------------------+
|First_name|               Email|     encrypted email|                Maid|
+----------+--------------------+--------------------+--------------------+
|     Laura|jadecook@example.org|8917ee04612ee225c...|f5fba91fcaed7bf81...|
|     Laura|jadecook@example.org|8917ee04612ee225c...|e0c625bc15196f574...|
|     Laura|loweryjennifer@ex...|2a558d6527d060579...|f5fba91fcaed7bf81...|
|     Laura|loweryjennifer@ex...|2a558d6527d060579...|e0c625bc15196f574...|
|      Ryan|jadecook@example.org|8917ee04612ee225c...|f5fba91fcaed7bf81...|
|      Ryan|jadecook@example.org|8917ee04612ee225c...|e0c625bc15196f574...|
|      Ryan|loweryjennifer@ex...|2a558d6527d060579...|f5fba91fcaed7bf81...|
|      Ryan|loweryjennifer@ex...|2a558d6527d060579...|e0c625bc15196f574...|
+----------+--------------------+--------------------+--------------------+

